# 作业3：用SVM给Iris鸢尾花分类

SVM（支持向量机）就是找一个超平面把不同类分开，而且要让间隔最大。

一开始看公式头都大了...什么拉格朗日对偶、KKT条件，完全看不懂。
核函数那部分让AI帮我解释了一下大概知道是怎么回事，但数学推导还是有点懵。

简单理解：
- 线性SVM：直接画一条线（或超平面）分开
- 核函数：如果线性分不开，就把数据映射到高维空间再分
  - 线性核：就是线性SVM
  - 多项式核：用多项式做映射
  - RBF核：高斯核，最常用的
  - sigmoid核：类似神经网络

两个重要参数：
- C：惩罚系数，C越大越不容忍误分类
- gamma：RBF核的参数，gamma越大单个样本影响范围越小

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

## 1. 加载数据，先看看长什么样

In [ ]:
"""
运作流程：
    1. 用sklearn的datasets.load_iris()把鸢尾花数据集加载进来
    2. 把特征矩阵、标签、特征名、类别名分别存到变量里
    3. 打印数据集的基本信息，看看数据长什么样

重要变量：
    - iris: 加载的整个iris数据集对象
    - X: 特征矩阵，150个样本×4个特征（花萼长度/宽度、花瓣长度/宽度）
    - y: 标签向量，0/1/2分别对应三种花
    - feature_names: 四个特征的名字列表
    - target_names: 三种类别的名字列表

依赖关系：
    - sklearn.datasets.load_iris
    - numpy（np.bincount用来统计各类别样本数）
"""
iris = datasets.load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"数据集大小: {X.shape}")
print(f"特征名称: {feature_names}")
print(f"类别名称: {target_names}")
print(f"各类别样本数: {np.bincount(y)}")

In [ ]:
"""
运作流程：
    1. 创建2×3的子图画布，准备画6组特征对的散点图
    2. 遍历所有特征组合(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)
    3. 对每组特征，按类别用红绿蓝三种颜色画散点
    4. 加上坐标轴标签和图例，最后显示出来

重要变量：
    - fig, axes: 画布和2×3子图数组
    - pairs: 6组特征对的索引组合列表
    - colors: 三种类别对应的颜色['r','g','b']

依赖关系：
    - X, y, feature_names, target_names（来自数据加载cell）
    - matplotlib.pyplot
"""
# 画画散点图看看数据分布
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
colors = ['r', 'g', 'b']

for idx, (i, j) in enumerate(pairs):
    ax = axes[idx // 3][idx % 3]
    for c in range(3):
        ax.scatter(X[y==c, i], X[y==c, j], c=colors[c], label=target_names[c], alpha=0.7)
    ax.set_xlabel(feature_names[i])
    ax.set_ylabel(feature_names[j])
    ax.legend()

plt.suptitle('Iris数据集特征对散点图', fontsize=14)
plt.tight_layout()
plt.show()

# setosa明显和其他两类分得很开，versicolor和virginica有点重叠

## 2. 数据预处理

In [ ]:
"""
运作流程：
    1. 先把数据分成训练集和测试集，7:3的比例
    2. stratify=y保证训练集和测试集中各类别比例一致
    3. 用StandardScaler对训练集做标准化（fit+transform一起做）
    4. 对测试集用同样的参数做标准化（只transform，不能fit！不然就数据泄露了）

重要变量：
    - X_train, X_test: 划分后的训练/测试特征（还没标准化）
    - y_train, y_test: 划分后的训练/测试标签
    - scaler: StandardScaler标准化器，保存了训练集的均值和方差
    - X_train_scaled, X_test_scaled: 标准化后的训练/测试特征

依赖关系：
    - X, y（来自数据加载cell）
    - sklearn.model_selection.train_test_split
    - sklearn.preprocessing.StandardScaler
"""
# 划分训练集和测试集
# stratify=y的意思是让训练集和测试集里三种花的比例一样
# 不然万一训练集里全是setosa就完蛋了...
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 标准化！之前做线性回归就吃过没标准化的亏...
# SVM对特征的尺度也很敏感
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集大小: {X_train_scaled.shape}")
print(f"测试集大小: {X_test_scaled.shape}")

## 3. 试不同的核函数

In [ ]:
"""
运作流程：
    1. 定义4种核函数：linear（线性）、poly（多项式）、rbf（高斯）、sigmoid
    2. 对每种核函数，创建SVC模型（C=1.0）并用训练集训练
    3. 用训练好的模型预测测试集
    4. 计算准确率，把模型、预测结果、准确率都存到results字典里
    5. 打印每种核函数的准确率，方便对比

重要变量：
    - kernels: 核函数名称列表 ['linear','poly','rbf','sigmoid']
    - results: 字典，key是核函数名，value包含model/y_pred/accuracy
    - svm: 每轮循环中创建的SVC模型对象
    - acc: 每轮循环中计算的准确率

依赖关系：
    - X_train_scaled, y_train, X_test_scaled, y_test（来自数据预处理cell）
    - sklearn.svm.SVC
    - sklearn.metrics.accuracy_score
"""
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[kernel] = {
        'model': svm,
        'y_pred': y_pred,
        'accuracy': acc
    }
    print(f"核函数: {kernel:8s} | 准确率: {acc:.4f}")

# RBF核效果最好，sigmoid最差
# 不过Iris数据集比较好分，所以差距不是特别大

## 4. 混淆矩阵

In [ ]:
"""
运作流程：
    1. 创建1×4的子图，给4种核函数各画一个混淆矩阵
    2. 对每种核函数，用confusion_matrix算出混淆矩阵（3×3）
    3. 用imshow画热力图，蓝色深浅表示数量多少
    4. 在每个格子里写上具体的数字，数字颜色根据背景深浅自动调黑/白
    5. 设置坐标轴标签为类别名，横轴是预测值纵轴是真实值

重要变量：
    - cm: 混淆矩阵，3×3的numpy数组
    - fig, axes: 画布和1×4子图数组

依赖关系：
    - kernels, results（来自核函数训练cell）
    - target_names（来自数据加载cell）
    - sklearn.metrics.confusion_matrix
    - matplotlib.pyplot
"""
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for idx, kernel in enumerate(kernels):
    cm = confusion_matrix(y_test, results[kernel]['y_pred'])
    # 手动画热力图，不用seaborn了
    ax = axes[idx]
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(target_names, rotation=45, ha='right')
    ax.set_yticklabels(target_names)
    # 在每个格子里写数字
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
    ax.set_title(f'{kernel} kernel\nAcc={results[kernel]["accuracy"]:.4f}')
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')

plt.tight_layout()
plt.show()

## 5. 决策边界可视化（只用前两个特征）

因为4维特征没法画图，所以只取前两个特征来画决策边界。
不过这样准确率会降低，因为少了两个特征的信息。

In [ ]:
"""
运作流程：
    1. 只取前两个特征（花萼长度和花萼宽度），因为4维没法画图
    2. 重新划分训练集/测试集，再对2D数据做标准化
    3. 对4种核函数分别训练SVM模型
    4. 生成密集的网格点，对每个网格点做预测，用颜色表示预测类别
    5. 把训练数据点叠加上去，就能看到决策边界长什么样了

重要变量：
    - X_2d: 只含前两个特征的数据矩阵
    - X_train_2d_scaled, X_test_2d_scaled: 标准化后的2D训练/测试数据
    - y_train_2d, y_test_2d: 2D数据对应的标签
    - h: 网格步长（0.02），越小决策边界越平滑但计算越慢
    - grid: 所有网格点的坐标矩阵
    - Z: 网格点的预测结果，用来画背景色

依赖关系：
    - X, y, feature_names, target_names（来自数据加载cell）
    - kernels（来自核函数训练cell）
    - sklearn.svm.SVC
    - sklearn.preprocessing.StandardScaler
    - sklearn.model_selection.train_test_split
    - matplotlib.pyplot
"""
# 只取前两个特征
X_2d = X[:, :2]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.3, random_state=42, stratify=y
)

scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
X_test_2d_scaled = scaler_2d.transform(X_test_2d)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
colors = ['r', 'g', 'b']

for idx, kernel in enumerate(kernels):
    ax = axes[idx // 2][idx % 2]
    svm = SVC(kernel=kernel, C=1.0, random_state=42)
    svm.fit(X_train_2d_scaled, y_train_2d)
    
    # 画决策边界的方法：在网格上每个点都预测一下
    h = 0.02
    x_min = X_train_2d_scaled[:, 0].min() - 1
    x_max = X_train_2d_scaled[:, 0].max() + 1
    y_min = X_train_2d_scaled[:, 1].min() - 1
    y_max = X_train_2d_scaled[:, 1].max() + 1
    xx = []
    yy = []
    # 生成网格点
    for xi in np.arange(x_min, x_max, h):
        for yi in np.arange(y_min, y_max, h):
            xx.append(xi)
            yy.append(yi)
    xx = np.array(xx)
    yy = np.array(yy)
    # 把xx和yy拼成一个二维数组，每行是一个点的坐标，这样svm.predict才能用
    grid = np.column_stack((xx, yy))
    Z = svm.predict(grid)
    
    # 画背景色
    ax.scatter(xx, yy, c=Z, cmap='RdYlBu', alpha=0.1, s=1)
    # 画训练数据点
    for c in range(3):
        ax.scatter(X_train_2d_scaled[y_train_2d==c, 0],
                   X_train_2d_scaled[y_train_2d==c, 1],
                   c=colors[c], label=target_names[c],
                   edgecolors='k', s=30)
    
    acc = accuracy_score(y_test_2d, svm.predict(X_test_2d_scaled))
    ax.set_title(f'{kernel} kernel (Acc={acc:.4f})')
    ax.set_xlabel(feature_names[0] + ' (scaled)')
    ax.set_ylabel(feature_names[1] + ' (scaled)')
    ax.legend()

plt.suptitle('不同核函数的SVM决策边界（前两个特征）', fontsize=14)
plt.tight_layout()
plt.show()

# 可以看到RBF核的决策边界是弯曲的，能更好地适应数据
# 线性核就是直线分开
# sigmoid核效果确实不太好...

## 6. 看看C和gamma对RBF核的影响

In [ ]:
"""
运作流程：
    1. 固定gamma=1.0，遍历不同的C值（0.01, 0.1, 1, 10, 100）训练RBF核SVM
    2. 打印每个C值对应的准确率，看看C对模型的影响
    3. 固定C=1.0，遍历不同的gamma值（0.01, 0.1, 1, 10, 100）训练RBF核SVM
    4. 打印每个gamma值对应的准确率，看看gamma对模型的影响

重要变量：
    - C_values: 要测试的C值列表 [0.01, 0.1, 1, 10, 100]
    - gamma_values: 要测试的gamma值列表 [0.01, 0.1, 1, 10, 100]
    - acc: 每次实验计算出的准确率

依赖关系：
    - X_train_scaled, y_train, X_test_scaled, y_test（来自数据预处理cell）
    - sklearn.svm.SVC
    - sklearn.metrics.accuracy_score
"""
# 试不同的C值
C_values = [0.01, 0.1, 1, 10, 100]
print("不同C值（gamma=1.0）:")
for C in C_values:
    svm = SVC(kernel='rbf', C=C, gamma=1.0, random_state=42)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"  C={C:6.2f} | 准确率: {acc:.4f}")

print()

# 试不同的gamma值
gamma_values = [0.01, 0.1, 1, 10, 100]
print("不同gamma值（C=1.0）:")
for gamma in gamma_values:
    svm = SVC(kernel='rbf', C=1.0, gamma=gamma, random_state=42)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"  gamma={gamma:6.2f} | 准确率: {acc:.4f}")

# C太小→欠拟合，C太大→可能过拟合
# gamma太大→过拟合（决策边界太复杂），gamma太小→欠拟合

In [ ]:
"""
运作流程：
    1. 用双重循环遍历所有C和gamma的组合（5×5=25种）
    2. 对每个组合训练RBF核SVM，计算测试集准确率
    3. 把准确率存到acc_matrix矩阵里（行是C，列是gamma）
    4. 用imshow画热力图，颜色越深表示准确率越高
    5. 在每个格子里写上准确率的具体数字

重要变量：
    - acc_matrix: 准确率矩阵，5×5的numpy数组，行对应C值列对应gamma值
    - C_values, gamma_values: 来自上一个cell的参数列表

依赖关系：
    - X_train_scaled, y_train, X_test_scaled, y_test（来自数据预处理cell）
    - C_values, gamma_values（来自C和gamma影响cell）
    - sklearn.svm.SVC
    - sklearn.metrics.accuracy_score
    - matplotlib.pyplot
"""
# 画C和gamma的热力图
acc_matrix = np.zeros((len(C_values), len(gamma_values)))

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        svm = SVC(kernel='rbf', C=C, gamma=gamma, random_state=42)
        svm.fit(X_train_scaled, y_train)
        y_pred = svm.predict(X_test_scaled)
        acc_matrix[i, j] = accuracy_score(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(acc_matrix, cmap='YlOrRd')
ax.set_xticks(range(len(gamma_values)))
ax.set_yticks(range(len(C_values)))
ax.set_xticklabels(gamma_values)
ax.set_yticklabels(C_values)
ax.set_xlabel('gamma')
ax.set_ylabel('C')
# 在每个格子里写数字
for i in range(len(C_values)):
    for j in range(len(gamma_values)):
        ax.text(j, i, f'{acc_matrix[i, j]:.4f}', ha='center', va='center')
plt.colorbar(im)
plt.title('RBF核SVM: C和gamma对准确率的影响')
plt.show()

## 7. 总结

In [ ]:
"""
运作流程：
    1. 打印各核函数的分类准确率汇总表

重要变量：
    - kernels: 核函数名称列表
    - results: 各核函数的准确率等结果字典

依赖关系：
    - kernels, results（来自核函数训练cell）
"""
# 各核函数SVM在Iris数据集上的分类准确率
print("=" * 40)
for kernel in kernels:
    print(f"  {kernel:8s}: {results[kernel]['accuracy']:.4f}")
print("=" * 40)